In [8]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib

# ==========================
# PARAMETERS
# ==========================

FRAME_SIZE = 0.025
HOP_SIZE = 0.01
N_FFT = 1024
LOW_FREQ = 400
HIGH_FREQ = 2000


# ==========================
# FEATURE EXTRACTION
# ==========================

def extract_frame_features(y, sr):

    frame_length = int(FRAME_SIZE * sr)
    hop_length = int(HOP_SIZE * sr)

    S = np.abs(librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    ))

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    band_mask = (freqs >= LOW_FREQ) & (freqs <= HIGH_FREQ)

    total_energy = np.sum(S, axis=0) + 1e-8
    band_energy = np.sum(S[band_mask, :], axis=0)

    band_ratio = band_energy / total_energy

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    psd = S / total_energy
    entropy = -np.sum(psd * np.log(psd + 1e-8), axis=0)

    features = np.vstack([
        band_ratio,
        centroid,
        flatness,
        rolloff,
        zcr,
        entropy
    ]).T

    return features, hop_length


# ==========================
# LABEL ASSIGNMENT
# ==========================

def create_frame_labels(label_file, total_frames, sr, hop_length):

    df = pd.read_csv(label_file,
                     sep="\t",
                     header=None,
                     names=["start", "end", "label"])

    labels = np.zeros(total_frames)

    for _, row in df.iterrows():
        if str(row["label"]).strip() == "Stridor":
            start_frame = int((row["start"] * sr) / hop_length)
            end_frame = int((row["end"] * sr) / hop_length)

            start_frame = max(0, start_frame)
            end_frame = min(total_frames, end_frame)

            labels[start_frame:end_frame] = 1

    return labels


# ==========================
# DATASET BUILDER
# ==========================

def build_dataset(audio_folder):

    X_all = []
    y_all = []

    audio_files = sorted(os.listdir(audio_folder))

    for file in tqdm(audio_files):

        if not file.endswith(".wav"):
            continue

        audio_path = os.path.join(audio_folder, file)

        base_name = file.replace(".wav", "")
        label_file = base_name + "_label_audacity.txt"
        label_path = os.path.join(audio_folder, label_file)

        if not os.path.exists(label_path):
            print(f"⚠ Missing label for {file}")
            continue

        y, sr = librosa.load(audio_path, sr=None)

        features, hop_length = extract_frame_features(y, sr)

        labels = create_frame_labels(
            label_path,
            features.shape[0],
            sr,
            hop_length
        )

        X_all.append(features)
        y_all.append(labels)

    X = np.vstack(X_all)
    y = np.hstack(y_all)

    return X, y

def post_process_predictions(predictions,
                             hop_size_sec,
                             min_duration_sec=0.15,
                             max_gap_frames=8):
    """
    Post-processing:
    1. Fill small gaps (<= max_gap_frames)
    2. Remove segments shorter than min_duration_sec
    """

    cleaned = predictions.copy()

    # ==========================
    # STEP 1: Fill small gaps
    # ==========================

    i = 0
    while i < len(cleaned):

        # Find start of zero gap
        if cleaned[i] == 0:
            gap_start = i

            while i < len(cleaned) and cleaned[i] == 0:
                i += 1

            gap_end = i
            gap_length = gap_end - gap_start

            # Check if gap is surrounded by ones
            if (gap_length <= max_gap_frames and
                gap_start > 0 and
                gap_end < len(cleaned) and
                cleaned[gap_start - 1] == 1 and
                cleaned[gap_end] == 1):

                cleaned[gap_start:gap_end] = 1

        else:
            i += 1

    # ==========================
    # STEP 2: Enforce minimum duration
    # ==========================

    min_frames = int(min_duration_sec / hop_size_sec)
    start = None

    for i in range(len(cleaned)):

        if cleaned[i] == 1 and start is None:
            start = i

        if (cleaned[i] == 0 or i == len(cleaned)-1) and start is not None:

            end = i if cleaned[i] == 0 else i + 1

            if end - start < min_frames:
                cleaned[start:end] = 0

            start = None

    return cleaned

# ==========================
# MAIN
# ==========================

def main():

    audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Stridor count"

    print("Building dataset...")
    X, y = build_dataset(audio_folder)

    feature_names = [
        "band_ratio",
        "centroid",
        "flatness",
        "rolloff",
        "zcr",
        "entropy"
    ]

    # Save full dataset
    full_df = pd.DataFrame(X, columns=feature_names)
    full_df["label"] = y
    full_df.to_csv("full_dataset.csv", index=False)

    print("Full dataset saved.")

    # Normalize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

    print("Training...")
    clf.fit(X_train, y_train)

    print("Predicting...")
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    # Apply 0.15 sec minimum duration constraint
    y_pred_clean = post_process_predictions(
        y_pred,
        hop_size_sec=HOP_SIZE,
        min_duration_sec=0.15,
        max_gap_frames=12
    )

    # ======================
    # SAVE TRAIN CSV
    # ======================

    train_df = pd.DataFrame(X_train, columns=feature_names)
    train_df["label"] = y_train
    train_df.to_csv("train_data.csv", index=False)

    # ======================
    # SAVE TEST CSV
    # ======================

    test_df = pd.DataFrame(X_test, columns=feature_names)
    test_df["label"] = y_test
    test_df["predicted_label"] = y_pred
    test_df["predicted_probability"] = y_prob

    test_df.to_csv("test_data_with_predictions.csv", index=False)

    # ======================
    # EVALUATION
    # ======================

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    joblib.dump(clf, "stridor_model.pkl")
    joblib.dump(scaler, "scaler.pkl")

    print("\nEverything saved successfully.")


if __name__ == "__main__":
    main()

Building dataset...


100%|██████████| 258/258 [00:05<00:00, 49.24it/s]


Full dataset saved.
Training...
Predicting...

Confusion Matrix:
[[29006   919]
 [ 6645  2156]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.81      0.97      0.88     29925
         1.0       0.70      0.24      0.36      8801

    accuracy                           0.80     38726
   macro avg       0.76      0.61      0.62     38726
weighted avg       0.79      0.80      0.77     38726


Everything saved successfully.


In [10]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# ==========================
# PARAMETERS
# ==========================

FRAME_SIZE = 0.025
HOP_SIZE = 0.01
N_FFT = 1024
LOW_FREQ = 400
HIGH_FREQ = 2000
MIN_DURATION = 0.15


# ==========================
# FEATURE EXTRACTION
# ==========================

def extract_features_with_time(y, sr):

    frame_length = int(FRAME_SIZE * sr)
    hop_length = int(HOP_SIZE * sr)

    S = np.abs(librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    ))

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    band_mask = (freqs >= LOW_FREQ) & (freqs <= HIGH_FREQ)

    total_energy = np.sum(S, axis=0) + 1e-8
    band_energy = np.sum(S[band_mask, :], axis=0)

    band_ratio = band_energy / total_energy
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    psd = S / total_energy
    entropy = -np.sum(psd * np.log(psd + 1e-8), axis=0)

    n_frames = len(band_ratio)

    start_times = np.arange(n_frames) * HOP_SIZE
    end_times = start_times + FRAME_SIZE

    features = np.vstack([
        band_ratio,
        centroid,
        flatness,
        rolloff,
        zcr,
        entropy
    ]).T

    return features, start_times, end_times, hop_length


# ==========================
# LABEL ASSIGNMENT
# ==========================

def create_frame_labels(label_file, total_frames, sr, hop_length):

    df = pd.read_csv(label_file,
                     sep="\t",
                     header=None,
                     names=["start", "end", "label"])

    labels = np.zeros(total_frames)

    for _, row in df.iterrows():
        if str(row["label"]).strip() == "Stridor":
            start_frame = int((row["start"] * sr) / hop_length)
            end_frame = int((row["end"] * sr) / hop_length)

            start_frame = max(0, start_frame)
            end_frame = min(total_frames, end_frame)

            labels[start_frame:end_frame] = 1

    return labels


# ==========================
# TEMPORAL CONSTRAINT
# ==========================

def post_process_predictions(predictions,
                             hop_size_sec,
                             min_duration_sec=0.15,
                             max_gap_frames=8):
    """
    Post-processing:
    1. Fill small gaps (<= max_gap_frames)
    2. Remove segments shorter than min_duration_sec
    """

    cleaned = predictions.copy()

    # ==========================
    # STEP 1: Fill small gaps
    # ==========================

    i = 0
    while i < len(cleaned):

        # Find start of zero gap
        if cleaned[i] == 0:
            gap_start = i

            while i < len(cleaned) and cleaned[i] == 0:
                i += 1

            gap_end = i
            gap_length = gap_end - gap_start

            # Check if gap is surrounded by ones
            if (gap_length <= max_gap_frames and
                gap_start > 0 and
                gap_end < len(cleaned) and
                cleaned[gap_start - 1] == 1 and
                cleaned[gap_end] == 1):

                cleaned[gap_start:gap_end] = 1

        else:
            i += 1

    # ==========================
    # STEP 2: Enforce minimum duration
    # ==========================

    min_frames = int(min_duration_sec / hop_size_sec)
    start = None

    for i in range(len(cleaned)):

        if cleaned[i] == 1 and start is None:
            start = i

        if (cleaned[i] == 0 or i == len(cleaned)-1) and start is not None:

            end = i if cleaned[i] == 0 else i + 1

            if end - start < min_frames:
                cleaned[start:end] = 0

            start = None

    return cleaned


# ==========================
# MAIN PIPELINE
# ==========================

def main():

    audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Stridor count"
    output_folder = "output"

    os.makedirs(output_folder + "/train", exist_ok=True)
    os.makedirs(output_folder + "/test", exist_ok=True)

    audio_files = sorted([f for f in os.listdir(audio_folder)
                          if f.endswith(".wav")])

    train_files, test_files = train_test_split(
        audio_files,
        test_size=0.2,
        random_state=42
    )

    X_train_all = []
    y_train_all = []

    # ======================
    # BUILD TRAIN SET
    # ======================

    for file in tqdm(train_files):

        base = file.replace(".wav", "")
        label_file = base + "_label_audacity.txt"

        y, sr = librosa.load(os.path.join(audio_folder, file), sr=None)

        features, start_t, end_t, hop_length = extract_features_with_time(y, sr)
        labels = create_frame_labels(
            os.path.join(audio_folder, label_file),
            features.shape[0],
            sr,
            hop_length
        )

        X_train_all.append(features)
        y_train_all.append(labels)

        df = pd.DataFrame(features, columns=[
            "band_ratio", "centroid", "flatness",
            "rolloff", "zcr", "entropy"
        ])

        df["start_time"] = start_t
        df["end_time"] = end_t
        df["true_label"] = labels

        df.to_csv(f"{output_folder}/train/{base}_train.csv", index=False)

    X_train = np.vstack(X_train_all)
    y_train = np.hstack(y_train_all)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    clf = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train, y_train)

    # ======================
    # TEST SET
    # ======================

    for file in tqdm(test_files):

        base = file.replace(".wav", "")
        label_file = base + "_label_audacity.txt"

        y, sr = librosa.load(os.path.join(audio_folder, file), sr=None)

        features, start_t, end_t, hop_length = extract_features_with_time(y, sr)
        labels = create_frame_labels(
            os.path.join(audio_folder, label_file),
            features.shape[0],
            sr,
            hop_length
        )

        X_test = scaler.transform(features)

        y_pred = clf.predict(X_test)
        y_pred_clean = post_process_predictions(
            y_pred,
            hop_size_sec=HOP_SIZE,
            min_duration_sec=0.15,
            max_gap_frames=12
        )

        df = pd.DataFrame(features, columns=[
            "band_ratio", "centroid", "flatness",
            "rolloff", "zcr", "entropy"
        ])

        df["start_time"] = start_t
        df["end_time"] = end_t
        df["true_label"] = labels
        df["predicted_label_raw"] = y_pred
        df["predicted_label_0.15s"] = y_pred_clean

        df.to_csv(f"{output_folder}/test/{base}_test.csv", index=False)

    joblib.dump(clf, "stridor_model.pkl")
    joblib.dump(scaler, "scaler.pkl")

    print("\n✅ File-wise CSV generation complete.")


if __name__ == "__main__":
    main()

  0%|          | 0/103 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:06<00:00,  3.91it/s]



✅ File-wise CSV generation complete.


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix


# ==========================
# PARAMETERS
# ==========================

HOP_SIZE = 0.01


# ==========================
# COUNT FROM LABEL FILE
# ==========================

def count_stridor_from_label(label_file):

    df = pd.read_csv(label_file,
                     sep="\t",
                     header=None,
                     names=["start", "end", "label"])

    count = df[df["label"].str.strip() == "Stridor"].shape[0]

    return count


# ==========================
# COUNT FROM PREDICTED FRAMES
# ==========================

def count_stridor_from_predictions(predicted_labels):

    count = 0
    in_segment = False

    for val in predicted_labels:
        if val == 1 and not in_segment:
            count += 1
            in_segment = True
        elif val == 0:
            in_segment = False

    return count


# ==========================
# MAIN
# ==========================

def main():

    test_csv_folder = "output/test"
    audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Stridor count"

    results = []

    # Accumulators for classification report
    y_true_all = []
    y_pred_raw_all = []
    y_pred_clean_all = []

    for csv_file in sorted(os.listdir(test_csv_folder)):

        if not csv_file.endswith("_test.csv"):
            continue

        base = csv_file.replace("_test.csv", "")
        label_file = os.path.join(audio_folder, base + "_label_audacity.txt")

        if not os.path.exists(label_file):
            print(f"⚠ Missing label file for {base}")
            continue

        df = pd.read_csv(os.path.join(test_csv_folder, csv_file))

        # ── Accumulate frame-level labels for classification report ──
        y_true_all.extend(df["true_label"].values)
        y_pred_raw_all.extend(df["predicted_label_raw"].values)
        y_pred_clean_all.extend(df["predicted_label_0.15s"].values)

        # ── Count-level evaluation ────────────────────────────────────
        true_count = count_stridor_from_label(label_file)

        pred_raw_count = count_stridor_from_predictions(
            df["predicted_label_raw"].values
        )

        pred_clean_count = count_stridor_from_predictions(
            df["predicted_label_0.15s"].values
        )

        results.append({
            "file": base,
            "true_count": true_count,
            "predicted_count_raw": pred_raw_count,
            "predicted_count_0.15s": pred_clean_count,
            "diff_raw": pred_raw_count - true_count,
            "diff_0.15s": pred_clean_count - true_count
        })

        print(f"{base}")
        print(f"  True count         : {true_count}")
        print(f"  Predicted          : {pred_clean_count}  (diff: {pred_clean_count - true_count:+d})")
        print()

    # ── Count summary ─────────────────────────────────────────────────
    results_df = pd.DataFrame(results)
    results_df.to_csv("stridor_count_comparison.csv", index=False)

    print("=" * 50)
    print(f"Total files evaluated : {len(results)}")
    print(f"Total true count      : {results_df['true_count'].sum()}")
    print(f"Total predicted       : {results_df['predicted_count_0.15s'].sum()}")
    print(f"Mean abs error        : {results_df['diff_0.15s'].abs().mean():.2f}")
    print("\nSaved to stridor_count_comparison.csv")

    # ── Frame-level classification report ────────────────────────────
    y_true  = np.array(y_true_all)
    y_pred_raw   = np.array(y_pred_raw_all)
    y_pred_clean = np.array(y_pred_clean_all)

    print("\n" + "=" * 50)
    print("CLASSIFICATION REPORT (raw predictions)")
    print("=" * 50)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred_raw))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_raw,
                                target_names=["No Stridor", "Stridor"]))

    print("=" * 50)
    print("CLASSIFICATION REPORT (post-processed, 0.15s min duration)")
    print("=" * 50)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred_clean))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_clean,
                                target_names=["No Stridor", "Stridor"]))


if __name__ == "__main__":
    main()

steth_20180901_10_19_28
  True count         : 3
  Predicted          : 3  (diff: +0)

steth_20180908_17_03_35
  True count         : 4
  Predicted          : 4  (diff: +0)

steth_20181018_13_41_52
  True count         : 3
  Predicted          : 4  (diff: +1)

steth_20190308_13_19_51
  True count         : 4
  Predicted          : 5  (diff: +1)

steth_20190308_13_20_19
  True count         : 3
  Predicted          : 3  (diff: +0)

trunc_2019-05-16-14-23-13-L4_2
  True count         : 3
  Predicted          : 10  (diff: +7)

trunc_2019-05-16-15-23-48-L8_4
  True count         : 4
  Predicted          : 3  (diff: -1)

trunc_2019-05-28-14-07-13-L1_3
  True count         : 3
  Predicted          : 1  (diff: -2)

trunc_2019-05-28-14-07-13-L1_8
  True count         : 4
  Predicted          : 4  (diff: +0)

trunc_2019-05-28-14-07-13-L2_3
  True count         : 3
  Predicted          : 8  (diff: +5)

trunc_2019-05-28-14-07-13-L4_1
  True count         : 4
  Predicted          : 13  (diff: +9)
